# optimizer-state-tensor-buffers — ex1: allocate a per-param zeros_like buffer at init

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `optimizer-state-tensor-buffers`. Running the final beacon cell reports progress against the `Optimizer: Per-param state buffers` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Optimizer: Per-param state buffers` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`optimizer-state-tensor-buffers`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "optimizer-state-tensor-buffers"
DD_SUBTOPIC = "Optimizer: Per-param state buffers"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Per-parameter state buffers — quick refresher

Optimizers that need MEMORY across steps (momentum, EMA, second moment) must keep a buffer FOR EACH parameter, allocated at construction time:

```
self.params = list(params)
self.b = [t.zeros_like(p) for p in self.params]    # one buffer per param
```

**Why `zeros_like` not `zeros`.** It mirrors `p`'s `shape`, `dtype`, AND `device` — so a buffer for a `(256, 768)` float16 CUDA weight is itself `(256, 768)` float16 on the same GPU. Initializing as `t.zeros(p.shape)` would silently put the buffer on CPU.

**Why a list, not one big tensor.** Different parameters can have different shapes; you can't flatten them into a single tensor without losing the per-param indexing that `step` relies on. PyTorch's own optimizers use the same per-param list pattern internally.

### Exercise 1 — allocate a per-param zeros_like buffer at init

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply `[t.zeros_like(p) for p in self.params]` to allocate a per-parameter momentum buffer at optimizer init time, matching the shape and dtype of every parameter.
> Keywords: zeros-like, per-param-buffer, init
> ```

**KCs targeted:** `state-buffer-allocated-via-zeros-like`, `state-buffer-one-per-param-as-list`

Implement `ex1_allocate_buffer(params)`. Return a Python list of zero-initialized tensors — one per param — each with the SAME shape AND dtype AND device AS the corresponding parameter.

Constraints:
- Use `t.zeros_like(p)`. Do NOT use `t.zeros(p.shape)` (that loses dtype/device).
- The result is a list, NOT a single stacked tensor — different params can have different shapes.
- Buffers do NOT track gradients (`zeros_like` returns `requires_grad=False` by default — verify that).

Input:
- `params`: iterable of leaf tensors (typically from `model.parameters()`).

The test passes a mix of `(256, 768)`, `(10,)`, `(3, 3, 5)` parameters and verifies shape, dtype, requires_grad, and that every entry is all-zeros.

In [ ]:
def ex1_allocate_buffer(params) -> list:
    """Return [zeros_like(p) for p in params] — one buffer per param."""
    raise NotImplementedError()


def _test_ex1():
    # Build a diverse param list: different shapes + a float64 entry.
    p1 = t.randn(256, 768, requires_grad=True)
    p2 = t.randn(10, requires_grad=True)
    p3 = t.randn(3, 3, 5, requires_grad=True)
    p4 = t.randn(4, 4, dtype=t.float64, requires_grad=True)
    params = [p1, p2, p3, p4]

    buffers = ex1_allocate_buffer(params)

    assert isinstance(buffers, list), f'must return a list, got {type(buffers)}'
    assert len(buffers) == 4, f'expected 4 buffers, got {len(buffers)}'

    for i, (p, b) in enumerate(zip(params, buffers)):
        assert b.shape == p.shape, (
            f'buffers[{i}] shape {tuple(b.shape)} != param shape {tuple(p.shape)}'
        )
        assert b.dtype == p.dtype, (
            f'buffers[{i}] dtype {b.dtype} != param dtype {p.dtype}; '
            f'did you use t.zeros(p.shape) instead of t.zeros_like(p)?'
        )
        assert b.requires_grad is False, (
            f'buffers[{i}].requires_grad must be False; '
            f'state buffers are NOT trainable'
        )
        assert t.all(b == 0), f'buffers[{i}] must be all zeros'

    # Each buffer must be a DISTINCT tensor (mutating one mustn't affect another).
    buffers[0] += 1.0
    for i in (1, 2, 3):
        assert t.all(buffers[i] == 0), (
            f'buffers[{i}] aliased buffers[0]; did you reuse the same tensor?'
        )

    # Buffer must not alias its parameter — modifying buffer[0] left p1 untouched.
    assert not t.allclose(p1, buffers[0]), (
        'param and buffer should be independent storage; '
        'did you accidentally return the param list itself?'
    )
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_allocate_buffer(params):
    return [t.zeros_like(p) for p in params]
```

**Why `zeros_like` and not `zeros`.** `zeros_like(p)` mirrors `p`'s `shape`, `dtype`, `layout`, and `device`. For a `(256, 768)` `float16` `cuda:0` weight, the resulting buffer is `(256, 768)` `float16` `cuda:0`. `t.zeros(p.shape)` would give you `float32` on CPU — wrong dtype causes silent precision drift in the optimizer state; wrong device crashes the first buffer arithmetic operation.

**Why a list of tensors rather than one big tensor.** Different params have different shapes. A `Linear(3, 5)` has `(5, 3)` weight and `(5,)` bias — those can't share a tensor. PyTorch internally also stores per-param state as a dict keyed by param id, mapping to per-param buffers.

**`zeros_like` default for `requires_grad`.** False, which is what we want — buffers are bookkeeping, not parameters to optimize through.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()